# Parallelization — Oil & Gas Compressor Fleet Screening

## Pattern
Parallelization sends the **same prompt to multiple independent inputs concurrently** using thread pools. This is the right pattern when:
- The same analysis must be applied to many independent items
- Items don't depend on each other's results
- Latency matters — sequential processing is too slow at field scale

## O&G Use Case: Compressor Fleet Health Screening
A midstream operator runs 8 gas compressors across a gathering network. Each shift, maintenance engineers must screen all units for early signs of failure — but reviewing 8 compressor data sheets sequentially takes hours.

With parallelization, all 8 units are screened **simultaneously**. Each compressor gets the same expert analysis prompt applied to its specific operating data. Results come back in the time it takes to analyse one unit.

**Typical O&G applications:**
- Fleet-wide equipment screening (compressors, pumps, separators)
- Multi-well production optimization across a portfolio
- Parallel regulatory compliance checks across all contracts
- Simultaneous stakeholder impact analysis for project decisions

In [ ]:
%pip install openai -q

In [ ]:
import sys
try:
    _nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    sys.path.insert(0, '/Workspace' + _nb_path.rsplit('/', 1)[0])
except Exception:
    sys.path.insert(0, '.')

from concurrent.futures import ThreadPoolExecutor, as_completed
from util import llm_call

def screen_fleet(prompt: str, units: list[dict], n_workers: int = 4) -> list[dict]:
    """Run the same screening prompt against all units concurrently.
    
    Args:
        prompt: The analysis prompt to apply to each unit.
        units: List of dicts with 'id' and 'data' keys.
        n_workers: Max concurrent threads (default 4 — safe for Databricks serverless).
    
    Returns:
        List of dicts with unit id, analysis, and status.
    """
    def analyse_unit(unit):
        result = llm_call(f"{prompt}\n\nUnit: {unit['id']}\nOperating Data:\n{unit['data']}")
        return {'id': unit['id'], 'analysis': result}

    print(f"Screening {len(units)} units with {n_workers} parallel workers...\n")
    results = []

    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        futures = {executor.submit(analyse_unit, unit): unit['id'] for unit in units}
        for future in as_completed(futures):
            unit_id = futures[future]
            result = future.result()
            print(f"  ✓ {unit_id} complete")
            results.append(result)

    # Sort back to original order for consistent output
    id_order = [u['id'] for u in units]
    results.sort(key=lambda r: id_order.index(r['id']))
    return results

## Mock Compressor Fleet Data
Eight compressors across a Permian Basin gathering network. Data simulates what would come from a SCADA historian or daily rounds sheet — suction/discharge pressures, temperatures, vibration, power draw, and operator notes.

In [ ]:
COMPRESSOR_FLEET = [
    {
        "id": "COMP-101",
        "data": """
        Location: Pad A - Inlet Compression
        Type: Reciprocating, 2-stage, 500 HP
        Runtime hours: 12,450
        Last PM: 45 days ago (scheduled at 90 days)
        Suction pressure: 85 psi (design: 80-100 psi)
        Discharge pressure: 1,050 psi (design: 900-1,100 psi)
        Suction temp: 72°F  |  Discharge temp: 285°F (design: <310°F)
        Vibration: 0.18 in/s (alarm: 0.30 in/s)
        Power draw: 412 kW (rated: 373 kW)
        Lube oil pressure: 48 psi (min: 40 psi)
        Lube oil temp: 198°F (max: 220°F)
        Packing leaks: None reported
        Operator note: Running slightly high on power, been like this 2 weeks
        """
    },
    {
        "id": "COMP-102",
        "data": """
        Location: Pad A - Booster
        Type: Reciprocating, single-stage, 300 HP
        Runtime hours: 6,820
        Last PM: 12 days ago
        Suction pressure: 92 psi (design: 80-100 psi)
        Discharge pressure: 480 psi (design: 450-550 psi)
        Suction temp: 68°F  |  Discharge temp: 241°F (design: <280°F)
        Vibration: 0.09 in/s (alarm: 0.30 in/s)
        Power draw: 228 kW (rated: 224 kW)
        Lube oil pressure: 52 psi (min: 40 psi)
        Lube oil temp: 187°F (max: 220°F)
        Packing leaks: None reported
        Operator note: No issues, running clean
        """
    },
    {
        "id": "COMP-103",
        "data": """
        Location: Pad B - Inlet Compression
        Type: Reciprocating, 2-stage, 500 HP
        Runtime hours: 18,200
        Last PM: 78 days ago (scheduled at 90 days)
        Suction pressure: 77 psi (design: 80-100 psi) <- LOW
        Discharge pressure: 1,120 psi (design: 900-1,100 psi) <- HIGH
        Suction temp: 74°F  |  Discharge temp: 318°F (design: <310°F) <- OVER TEMP
        Vibration: 0.27 in/s (alarm: 0.30 in/s) <- APPROACHING ALARM
        Power draw: 468 kW (rated: 373 kW) <- 25% OVER RATED
        Lube oil pressure: 41 psi (min: 40 psi) <- NEAR MINIMUM
        Lube oil temp: 214°F (max: 220°F) <- NEAR MAXIMUM
        Packing leaks: Rod packing on cylinder 2 showing slight seep
        Operator note: Unit sounds rougher than normal, some knocking on startup
        """
    },
    {
        "id": "COMP-104",
        "data": """
        Location: Pad B - Booster
        Type: Centrifugal, 750 HP
        Runtime hours: 9,100
        Last PM: 30 days ago
        Suction pressure: 465 psi (design: 450-500 psi)
        Discharge pressure: 1,200 psi (design: 1,100-1,300 psi)
        Suction temp: 95°F  |  Discharge temp: 198°F (design: <220°F)
        Vibration - DE bearing: 0.11 in/s (alarm: 0.25 in/s)
        Vibration - NDE bearing: 0.13 in/s (alarm: 0.25 in/s)
        Power draw: 601 kW (rated: 560 kW)
        Seal gas flow: 2.1 scfm (normal: 1.8-2.5 scfm)
        Lube oil pressure: 58 psi (min: 45 psi)
        Lube oil temp: 172°F (max: 200°F)
        Operator note: Slight increase in power draw noted over last 10 days
        """
    },
    {
        "id": "COMP-105",
        "data": """
        Location: Pad C - Inlet Compression
        Type: Reciprocating, 2-stage, 500 HP
        Runtime hours: 3,200
        Last PM: 5 days ago (new installation, commissioning complete)
        Suction pressure: 88 psi (design: 80-100 psi)
        Discharge pressure: 975 psi (design: 900-1,100 psi)
        Suction temp: 70°F  |  Discharge temp: 267°F (design: <310°F)
        Vibration: 0.07 in/s (alarm: 0.30 in/s)
        Power draw: 358 kW (rated: 373 kW)
        Lube oil pressure: 55 psi (min: 40 psi)
        Lube oil temp: 182°F (max: 220°F)
        Packing leaks: None
        Operator note: New unit, running beautifully
        """
    },
    {
        "id": "COMP-106",
        "data": """
        Location: Pad C - Booster
        Type: Reciprocating, single-stage, 300 HP
        Runtime hours: 14,600
        Last PM: 102 days ago (OVERDUE — scheduled at 90 days)
        Suction pressure: 89 psi (design: 80-100 psi)
        Discharge pressure: 492 psi (design: 450-550 psi)
        Suction temp: 71°F  |  Discharge temp: 255°F (design: <280°F)
        Vibration: 0.21 in/s (alarm: 0.30 in/s)
        Power draw: 241 kW (rated: 224 kW)
        Lube oil pressure: 44 psi (min: 40 psi)
        Lube oil temp: 208°F (max: 220°F)
        Packing leaks: Valve cover on cylinder 1 - minor weep
        Operator note: PM is overdue, keep meaning to schedule it
        """
    },
    {
        "id": "COMP-107",
        "data": """
        Location: Central Facility - Sales Compression
        Type: Centrifugal, 1,200 HP
        Runtime hours: 22,800
        Last PM: 60 days ago
        Suction pressure: 950 psi (design: 900-1,000 psi)
        Discharge pressure: 1,480 psi (design: 1,400-1,600 psi)
        Suction temp: 88°F  |  Discharge temp: 187°F (design: <210°F)
        Vibration - DE bearing: 0.19 in/s (alarm: 0.25 in/s)
        Vibration - NDE bearing: 0.22 in/s (alarm: 0.25 in/s) <- NEAR ALARM
        Power draw: 980 kW (rated: 895 kW)
        Seal gas flow: 3.8 scfm (normal: 2.5-4.0 scfm)
        Lube oil pressure: 62 psi (min: 45 psi)
        Lube oil temp: 178°F (max: 200°F)
        Operator note: NDE vibration has been creeping up for 3 weeks
        """
    },
    {
        "id": "COMP-108",
        "data": """
        Location: Central Facility - Standby
        Type: Reciprocating, 2-stage, 500 HP
        Runtime hours: 8,900
        Last PM: 20 days ago
        Status: STANDBY (hot standby, ready to start)
        Last run: 8 days ago (weekly exercise run - 2 hours)
        Suction pressure: 85 psi (pressured up, design: 80-100 psi)
        Discharge pressure: N/A (not running)
        Vibration: N/A (not running)
        Power draw: 0 kW
        Lube oil pressure: 12 psi (pre-lube pump active, min running: 40 psi)
        Lube oil temp: 95°F (ambient - heater on)
        Operator note: Standby unit in good shape, exercised on schedule
        """
    }
]

print(f"Fleet loaded: {len(COMPRESSOR_FLEET)} compressors")

## Screening Prompt
One prompt applied to all units in parallel. The prompt encodes the engineering judgment — what to look for, how to score it, what to recommend.

In [ ]:
SCREENING_PROMPT = """You are a rotating equipment engineer performing a daily compressor health screen.

Analyse the operating data for this compressor unit and produce a structured health assessment.

Evaluate these areas:
1. Pressures — suction/discharge vs. design range
2. Temperatures — discharge temp and lube oil temp vs. limits
3. Vibration — current level vs. alarm setpoint, trend if noted
4. Power draw — actual vs. rated (>10% over = flag, >20% over = critical)
5. Lube system — oil pressure and temperature vs. limits
6. Mechanical condition — leaks, noise, abnormal observations
7. Maintenance status — PM currency, overdue items

Output format (use exactly):

HEALTH STATUS: [GREEN / AMBER / RED]
OVERALL RISK: [LOW / MEDIUM / HIGH / CRITICAL]

KEY FINDINGS:
- [Finding 1 with specific value and deviation]
- [Finding 2...]

RECOMMENDED ACTIONS:
1. [Action with urgency: IMMEDIATE / THIS SHIFT / THIS WEEK / NEXT PM]
2. [...]

EARLIEST SAFE FAILURE MODE IF NOT ACTIONED: [brief description]

Be concise and specific. Use actual values from the data."""

## Run Parallel Fleet Screen

In [ ]:
import time

start = time.time()
results = screen_fleet(SCREENING_PROMPT, COMPRESSOR_FLEET, n_workers=4)
elapsed = time.time() - start

print(f"\nAll {len(results)} units screened in {elapsed:.1f}s")

## Individual Unit Results

In [ ]:
for r in results:
    print(f"\n{'='*60}")
    print(f"  {r['id']}")
    print('='*60)
    print(r['analysis'])

## Fleet Summary Dashboard
Roll up individual assessments into a priority-sorted fleet view.

In [ ]:
SUMMARY_PROMPT = """You are a maintenance supervisor reviewing compressor health assessments from your shift engineer.

Below are health assessments for all units in the fleet. Produce a one-page fleet summary for the operations manager.

Format:
FLEET HEALTH SUMMARY
====================
Units screened: X  |  RED: X  |  AMBER: X  |  GREEN: X

PRIORITY ACTION LIST (most urgent first):
  1. [Unit] — [action] — [urgency]
  ...

UNIT STATUS TABLE:
  [Unit ID] | [Status] | [Risk] | [Top concern]
  ...

MAINTENANCE SCHEDULING RECOMMENDATIONS:
  [Which units need PM scheduled, in what order]

Keep it under 250 words. This is for the ops manager morning briefing."""

# Combine all assessments into one input for the summary
combined = "\n\n".join([f"--- {r['id']} ---\n{r['analysis']}" for r in results])
summary = llm_call(f"{SUMMARY_PROMPT}\n\nAssessments:\n{combined}")

print("\n" + "="*60)
print("  FLEET SUMMARY")
print("="*60)
print(summary)

## Key Takeaways

**Why parallelization instead of sequential?**

| Approach | Time for 8 units | Time for 100 units |
|---|---|---|
| Sequential | ~8× single call | ~100× single call |
| Parallel (4 workers) | ~2× single call | ~25× single call |
| Parallel (8 workers) | ~1× single call | ~13× single call |

At field scale — a gathering system with 50+ compressors — sequential screening is impractical before the next shift starts. Parallelization makes it a scheduled job that completes in minutes.

**The pattern works whenever:**
- The same analysis applies to many independent items
- Items don't depend on each other
- You want results in minutes, not hours

**Where to go from here:**
- Replace mock data with live SCADA historian queries (Databricks + OSIsoft PI connector)
- Write results to a Delta table for trending and alerting
- Schedule as a Databricks Job every shift using the DABs config in this repo
- Add a Slack/Teams notification for RED units